# CP5 · Examen blanc de la partie V — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_5/03_examen_solutions.ipynb)

L'enregistrement des réponses que vérifie le notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`), calculées à partir des nombres du sujet, avec NumPy et des fractions exactes, et recoupées par des vérifications (`assert`) ; les erreurs classiques reçoivent un message qui dit où chercher. Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les réponses vérifiées après l'examen

In [2]:
# The answers are computed from the numbers of the statements (01_examen_sujet.md), with NumPy and exact fractions
import math
from fractions import Fraction
from itertools import product

import numpy as np


def ints(a):
    """An array (or nested lists) of whole numbers as lists of int, after checking that every value is whole."""
    arr = np.asarray(a, dtype=float)
    assert np.allclose(arr, np.round(arr)), arr
    return np.round(arr).astype(int).tolist()


def floats(a):
    """Fractions or floats (a number, nested lists or an array) as plain floats."""
    return np.asarray(a, dtype=float).tolist()


def fmat(rows):
    """A matrix written with the decimals of the statement ("0.5"), as exact fractions."""
    return [[Fraction(x) for x in row] for row in rows]


def fmatmul(p, q):
    return [[sum(a * b for a, b in zip(row, col)) for col in zip(*q)] for row in p]


def fmatvec(p, v):
    return [sum(a * b for a, b in zip(row, v)) for row in p]


def fpow(p, k):
    out = [[Fraction(int(i == j)) for j in range(len(p))] for i in range(len(p))]
    for _ in range(k):
        out = fmatmul(p, out)                          # one product per step
    return out


def ftranspose(p):
    return [list(col) for col in zip(*p)]


def finv2(p):
    """The inverse of a 2 x 2 matrix of fractions."""
    (a, b), (c, d) = p
    det = a * d - b * c
    return [[d / det, -b / det], [-c / det, a / det]]


def fsqrt(q):
    """The exact square root of a fraction that is a perfect square (16/25 -> 4/5)."""
    q = Fraction(q)
    num, den = math.isqrt(q.numerator), math.isqrt(q.denominator)
    assert num * num == q.numerator and den * den == q.denominator, q
    return Fraction(num, den)


def eig2(p):
    """The two eigenvalues of a 2 x 2 matrix of fractions when they are rational: from the trace and the determinant."""
    tr, det = p[0][0] + p[1][1], p[0][0] * p[1][1] - p[0][1] * p[1][0]
    root = fsqrt(tr * tr - 4 * det)
    return sorted([(tr - root) / 2, (tr + root) / 2])


def norm_1(p):
    return max(sum(abs(x) for x in col) for col in zip(*p))


def norm_inf(p):
    return max(sum(abs(x) for x in row) for row in p)


def rho(m):
    return float(np.abs(np.linalg.eigvals(np.asarray(m, dtype=float))).max())


def behaviour(m, steps=600):
    """1: the powers tend to 0; 2: bounded without tending to 0; 3: not bounded (read on the first powers)."""
    m = np.asarray(m, dtype=float)
    power, norms = np.eye(len(m)), []
    for _ in range(steps):
        power = m @ power
        norms.append(np.linalg.norm(power, 2))
    if norms[-1] < 1e-9:
        return 1
    return 2 if max(norms) < 1e3 else 3


def as_pi(angle):
    """An angle that is a rational multiple of pi, written exactly: "pi/4", "-3*pi/4"."""
    q = Fraction(angle / math.pi).limit_denominator(24)
    assert abs(float(q) * math.pi - angle) < 1e-9, angle
    top = {1: "pi", -1: "-pi"}.get(q.numerator, f"{q.numerator}*pi")
    return top if q.denominator == 1 else f"{top}/{q.denominator}"


def toeplitz(w, n):
    """The (n + k - 1, n) matrix of y -> y * w (full mode): T_ij = w_(i-j), 0 outside the kernel."""
    k = len(w)
    return np.array([[w[i - j] if 0 <= i - j < k else 0 for j in range(n)] for i in range(n + k - 1)])


# CP5.1 (1): the polynomials P and Q, from the constant term up
A_1, B_1 = np.array([2, 3, -1]), np.array([1, 0, -2, 1])
AB_1 = np.convolve(A_1, B_1)                               # the coefficients of PQ
SLID_1 = np.convolve(A_1, B_1[::-1])                       # b slid along a without being reversed
SIGNS_1 = (-1) ** np.arange(len(AB_1))
ALT_1 = int(SIGNS_1 @ AB_1)
P_1, Q_1 = (int(np.polynomial.polynomial.polyval(-1, c)) for c in (A_1, B_1))
assert ALT_1 == P_1 * Q_1                                  # the value of PQ at X = -1
SUM_1 = int(AB_1.sum())                                    # PQ at X = 1, no alternation

# CP5.1 (2): the signal x and the kernel w
X_1, W_1 = np.array([3, -2, 4, 1, 0, -1]), np.array([3, 0, -2, 1])
N_1, K_1 = len(X_1), len(W_1)
CORR_1 = np.correlate(X_1, W_1, "valid")                   # (w ⋆ x)_i = Σ_u w_u x_(i+u)
assert CORR_1.tolist() == [sum(W_1[u] * X_1[i + u] for u in range(K_1)) for i in range(N_1 - K_1 + 1)]
VALID_1 = np.convolve(X_1, W_1, "valid")                   # the kernel reversed: a convolution
FULL_1 = np.convolve(X_1, W_1)
assert FULL_1.tolist() == [sum(X_1[i] * W_1[m - i] for i in range(N_1) if 0 <= m - i < K_1)
                           for m in range(N_1 + K_1 - 1)]
FULLCORR_1 = np.convolve(X_1, W_1[::-1])                   # a full correlation: the kernel not reversed
SAME_1 = np.convolve(X_1, W_1, "same")
START_1 = (min(N_1, K_1) - 1) // 2
assert SAME_1.tolist() == FULL_1[START_1:START_1 + N_1].tolist()
T3_1 = toeplitz(W_1, 3)
assert all((T3_1 @ y).tolist() == np.convolve(y, W_1).tolist() for y in np.eye(3, dtype=int))
T40_1 = toeplitz(W_1, 40)
NNZ_1 = int(np.count_nonzero(T40_1))
T6_1 = toeplitz(W_1, N_1)
V_1 = T6_1[K_1 - 1:N_1]                                    # valid mode: the rows of the complete windows
assert (V_1 @ X_1).tolist() == VALID_1.tolist()



# CP5.2 (1): the windows of k = 9 weights on n = 472 samples
def positions(n, k, s=1, p=0, d=1):
    """The number of positions of a window of k weights (stride s, padding p, dilation d) on n cases."""
    return (n + 2 * p - d * (k - 1) - 1) // s + 1


def positions_counted(n, k, s=1, p=0, d=1):
    """The same number, by brute force: the windows whose last case stays in the padded signal."""
    return sum(1 for i in range(n + 2 * p) if s * i + d * (k - 1) <= n + 2 * p - 1)


N_2, K_2, S_2 = 472, 9, 4
POS_A_2 = positions(N_2, K_2, S_2)
POS_B_2 = positions(N_2, K_2, S_2, p=5, d=3)
assert POS_A_2 == positions_counted(N_2, K_2, S_2) and POS_B_2 == positions_counted(N_2, K_2, S_2, p=5, d=3)
CEIL_A_2 = -(-(N_2 - K_2) // S_2) + 1                      # rounded up
NODIL_B_2 = positions(N_2, K_2, S_2, p=5)
B_SLIPS_2 = {(N_2 + 5 - 3 * (K_2 - 1) - 1) // S_2 + 1,      # the padding counted once
             (N_2 + 10 - 3 * K_2) // S_2 + 1,              # the span d k
             (N_2 + 10 - 3 * (K_2 - 1) - 1) // S_2}        # the + 1 forgotten
B_UP_2 = {-(-(N_2 + 10 - 3 * (K_2 - 1) - 1) // S_2) + 1, POS_A_2}   # rounded up, or neither p nor d
NOPAD_B_2 = positions(N_2, K_2, S_2, d=3)
WINDOWS_2 = [k for k in range(1, N_2 + 1) if positions(N_2, k, S_2) == 110]       # the window lengths k
assert all(positions_counted(N_2, k, S_2) == 110 for k in WINDOWS_2)
NO_ONE_2 = [k for k in range(1, N_2 + 1) if (N_2 - k) // S_2 == 110]             # the + 1 forgotten
NO_MINUS_2 = [k for k in range(1, N_2 + 1) if (N_2 - k + 1) // S_2 + 1 == 110]   # the − 1 forgotten: n − k + 1
OUT_2 = -(-N_2 // S_2)                                     # TensorFlow and Keras: ceil(n / s)
TOTAL_2 = max((OUT_2 - 1) * S_2 + K_2 - N_2, 0)            # d = 1: (n_out − 1) s + d(k − 1) + 1 − n zeros
SAME_2 = [OUT_2, TOTAL_2 // 2, TOTAL_2 - TOTAL_2 // 2]
assert (N_2 + TOTAL_2 - (K_2 - 1) - 1) // S_2 + 1 == OUT_2 > (N_2 + TOTAL_2 - 1 - (K_2 - 1) - 1) // S_2 + 1

# CP5.2 (2): the transposed convolution of y by w with a stride 2
Y_2, WT_2, ST_2 = np.array([1, -2, 3, 2]), np.array([3, -2, 2]), 2
UP_2 = np.zeros((len(Y_2) - 1) * ST_2 + 1, dtype=int)
UP_2[::ST_2] = Y_2                                         # s − 1 zeros inserted between the values
TRANS_2 = np.convolve(UP_2, WT_2)                          # then the full convolution by w
STAMPS_2 = np.zeros(len(TRANS_2), dtype=int)
OVERWRITTEN_2 = np.zeros(len(TRANS_2), dtype=int)
for i, y in enumerate(Y_2):                                # each y_i prints y_i w from the case s i
    STAMPS_2[ST_2 * i:ST_2 * i + len(WT_2)] += y * WT_2
    OVERWRITTEN_2[ST_2 * i:ST_2 * i + len(WT_2)] = y * WT_2
C_2 = np.array([[WT_2[j - ST_2 * i] if 0 <= j - ST_2 * i < len(WT_2) else 0 for j in range(len(TRANS_2))]
                for i in range(len(Y_2))])                 # the sliding sum of stride 2 by w, on 9 cases
assert STAMPS_2.tolist() == TRANS_2.tolist() == (C_2.T @ Y_2).tolist()
REVERSED_2 = np.convolve(UP_2, WT_2[::-1])


def transposed_size(m, k, s, p=0, d=1, p_out=0):
    return (m - 1) * s - 2 * p + d * (k - 1) + 1 + p_out


SIZE_2 = transposed_size(50, 5, 3, p=2, p_out=1)
assert positions(SIZE_2, 5, 3, p=2) == 50                  # the sliding sum of the same settings gives back 50
SIZE_SLIPS_2 = {"no p_out": transposed_size(50, 5, 3, p=2), "p added": 49 * 3 + 2 * 2 + 4 + 1 + 1,
                "p once": transposed_size(50, 5, 3, p=1, p_out=1), "p ignored": transposed_size(50, 5, 3, p_out=1)}
KH_2, SH_2, MH_2 = 10, 3, 30                               # k = 10, s = 3, on 30 input values
STAMPS_COUNT_2 = [sum(1 for i in range(MH_2) if 0 <= j - SH_2 * i <= KH_2 - 1) for j in range((MH_2 - 1) * SH_2 + KH_2)]
COUNTS_2 = sorted(set(STAMPS_COUNT_2[KH_2:(MH_2 - 1) * SH_2]))   # the interior cases, far from both edges

# CP5.3 (1): the grid G and the kernel K, a cross-correlation in valid mode
G_3 = np.array([[3, 1, 0, 2, 4], [1, 4, 2, 0, 1], [2, 0, 3, 1, 2], [0, 2, 1, 3, 1]])
K_3 = np.array([[2, -2, 6], [1, -1, 3], [-1, 1, -3]])
assert np.linalg.matrix_rank(K_3) == 1
B_3 = K_3[0] / K_3[0, 0]                                   # b_0 = 1: the row 0 divided by K_00
A_3 = K_3[:, 0] / B_3[0]
assert (np.outer(A_3, B_3) == K_3).all()


def correlate2d(g, k):
    """Y_ij = Σ_u Σ_v K_uv G_(i+u, j+v), valid mode, no flip."""
    g, k = np.asarray(g), np.asarray(k)
    rows, cols = g.shape[0] - k.shape[0] + 1, g.shape[1] - k.shape[1] + 1
    return np.array([[np.sum(k * g[i:i + k.shape[0], j:j + k.shape[1]]) for j in range(cols)] for i in range(rows)])


Y_3 = correlate2d(G_3, K_3)
HORIZONTAL_3 = correlate2d(G_3, B_3[None, :])              # the horizontal pass by b, on every row of G
VERTICAL_3 = correlate2d(G_3, A_3[:, None])                # the vertical pass by a, on every column of G
assert (correlate2d(HORIZONTAL_3, A_3[:, None]) == Y_3).all() and (correlate2d(VERTICAL_3, B_3[None, :]) == Y_3).all()
CONV_3 = correlate2d(G_3, K_3[::-1, ::-1])                 # the 2D convolution: a half-turn of the kernel
TRANSPOSED_3 = correlate2d(G_3, K_3.T)
DIRECT_3 = Y_3.size * K_3.size                             # for the copy: one product per weight and per output
H_FIRST_3 = HORIZONTAL_3.size * len(B_3) + Y_3.size * len(A_3)   # the horizontal pass first, then the vertical one
V_FIRST_3 = VERTICAL_3.size * len(A_3) + Y_3.size * len(B_3)     # the other order
NARROW_3 = Y_3.size * len(B_3) + Y_3.size * len(A_3)       # the first pass on the rows of the output only
NOT_ONE_3 = int(HORIZONTAL_3.size * sum(abs(b) != 1 for b in B_3)
                + Y_3.size * sum(abs(a) != 1 for a in A_3))   # the multiplications by 1 or −1 skipped

# CP5.3 (2): the thermal camera, (5, 96, 144) -> 12 channels, kernels 3 x 5, bias, padding 1 in height and 2 in
# width, stride 2 along both axes
C_IN_3, C_OUT_3, H_3, W_3, KH_3, KW_3, PH_3, PW_3, S_3 = 5, 12, 96, 144, 3, 5, 1, 2, 2
PARAMS_3 = C_OUT_3 * (C_IN_3 * KH_3 * KW_3 + 1)


def out_shape(stride=S_3, ph=PH_3, pw=PW_3, kh=KH_3, kw=KW_3, c=C_OUT_3):
    """The shape (C, H, W) of the output, each axis counted with its own kernel size and padding."""
    return [c, positions(H_3, kh, stride, p=ph), positions(W_3, kw, stride, p=pw)]


SHAPE_3 = out_shape()
assert SHAPE_3[1:] == [positions_counted(H_3, KH_3, S_3, p=PH_3), positions_counted(W_3, KW_3, S_3, p=PW_3)]
SHAPE_SLIPS_3 = {"stride": out_shape(stride=1), "padding": out_shape(ph=0, pw=0), "channels": out_shape(c=C_IN_3),
                 "one side": [C_OUT_3, (H_3 + PH_3 - KH_3) // S_3 + 1, (W_3 + PW_3 - KW_3) // S_3 + 1],
                 "padding swapped": out_shape(ph=PW_3, pw=PH_3), "kernel swapped": out_shape(kh=KW_3, kw=KH_3),
                 "order": SHAPE_3[1:] + SHAPE_3[:1]}


def macs(rows, cols, c_out=C_OUT_3, c_in=C_IN_3):
    """One multiplication-addition per weight and per output position."""
    return rows * cols * c_out * c_in * KH_3 * KW_3


MACS_3 = macs(*SHAPE_3[1:])
NOSTRIDE_3 = macs(*SHAPE_SLIPS_3["stride"][1:])
NOPAD_3 = macs(*SHAPE_SLIPS_3["padding"][1:])
NO_CIN_3 = macs(*SHAPE_3[1:], c_in=1)
WITH_BIAS_3 = SHAPE_3[1] * SHAPE_3[2] * PARAMS_3
DEPTH_3, POINT_3 = C_IN_3 * (KH_3 * KW_3 + 1), C_OUT_3 * (C_IN_3 + 1)
SEP_3 = DEPTH_3 + POINT_3
NO_BIAS_3 = {DEPTH_3 - C_IN_3 + POINT_3, DEPTH_3 + POINT_3 - C_OUT_3, DEPTH_3 - C_IN_3 + POINT_3 - C_OUT_3}

# CP5.3 (3): the stack (k, s, d) of four sliding sums
STACK_3 = [(2, 1, 1), (4, 3, 1), (2, 1, 3), (3, 4, 1)]


def field(stack, dilation=True, own_stride=False):
    """r_l = r_(l-1) + d_l (k_l − 1) J_(l-1), J_l = J_(l-1) s_l (own_stride: J_l in place of J_(l-1))."""
    r, jump = 1, 1
    for k, s, d in stack:
        if own_stride:
            jump *= s
        r += (d if dilation else 1) * (k - 1) * jump
        if not own_stride:
            jump *= s
    return r


def read_cases(stack, output=0):
    """The cases of the original signal that the calculations of one output read, layer by layer, by brute force."""
    cases = {output}
    for k, s, d in reversed(stack):
        cases = {s * i + d * u for i in cases for u in range(k)}
    return cases


STRIDE_3 = math.prod(s for _, s, _ in STACK_3)
RF_3 = field(STACK_3)
assert max(read_cases(STACK_3)) - min(read_cases(STACK_3)) + 1 == RF_3 and min(read_cases(STACK_3, 1)) == STRIDE_3
KSUM_3 = sum(k for k, _, _ in STACK_3)                     # the sizes of the kernels added
SPANS_3 = sum(d * (k - 1) + 1 for k, _, d in STACK_3)      # the spans of the windows added
NOJUMP_3 = 1 + sum(d * (k - 1) for k, _, d in STACK_3)     # the strides of the previous layers forgotten
NOJUMP_NODIL_3 = 1 + sum(k - 1 for k, _, _ in STACK_3)     # the strides and the dilations forgotten

# CP5.4: five strain gauges on the five blades of a rotor, indices modulo 5; the largest INCREASE of z_1
X_4, H_4, EPS_4 = np.array([1, 3, -2, 0, 2]), np.array([1, -4, 2, 3, -2]), Fraction("0.2")
N_4 = len(X_4)
CIRC_4 = np.array([[H_4[(i - j) % N_4] for j in range(N_4)] for i in range(N_4)])
Z_4 = CIRC_4 @ X_4
assert Z_4.tolist() == [sum(X_4[i] * H_4[(m - i) % N_4] for i in range(N_4)) for m in range(N_4)]
FULL_4 = np.convolve(X_4, H_4)
assert Z_4.tolist() == [FULL_4[m] + (FULL_4[m + N_4] if m + N_4 < len(FULL_4) else 0) for m in range(N_4)]
UNFOLDED_4 = FULL_4[:N_4]                                  # the full convolution, nothing folded back
CIRC_CORR_4 = [sum(H_4[u] * X_4[(m + u) % N_4] for u in range(N_4)) for m in range(N_4)]   # h ⋆ x, made circular
PLUS_4 = [sum(X_4[i] * H_4[(m + i) % N_4] for i in range(N_4)) for m in range(N_4)]       # h_((m+i) mod n)
SHIFT_4 = np.array([[int(j == (i - 1) % N_4) for j in range(N_4)] for i in range(N_4)])  # (P x)_i = x_(i−1)
ZP2_4 = CIRC_4 @ (SHIFT_4 @ SHIFT_4 @ X_4)
assert ZP2_4.tolist() == (SHIFT_4 @ SHIFT_4 @ Z_4).tolist() and (CIRC_4 @ SHIFT_4 == SHIFT_4 @ CIRC_4).all()
BACK_4 = SHIFT_4.T @ SHIFT_4.T @ Z_4                       # shifted the other way
ONCE_4 = SHIFT_4 @ Z_4
ROW_4 = CIRC_4[1]                                          # z_1 = (row 1 of circ(h)) · x
DELTA_4 = [EPS_4 * int(np.sign(r)) for r in ROW_4]
WORST_INF_4 = EPS_4 * int(np.abs(ROW_4).sum())
VERTICES_4 = [(int(sum(r * e for r, e in zip(ROW_4, signs))), signs) for signs in product((-1, 1), repeat=N_4)]
assert max(VERTICES_4)[0] * EPS_4 == WORST_INF_4 and [EPS_4 * e for e in max(VERTICES_4)[1]] == DELTA_4
assert sum(v == max(VERTICES_4)[0] for v, _ in VERTICES_4) == 1   # a unique vertex: no weight is zero
WORST_TWO_4 = float(EPS_4) * float(np.linalg.norm(ROW_4))
H_SIGNS_4 = [EPS_4 * int(np.sign(h)) for h in H_4]
ROW0_SIGNS_4 = [EPS_4 * int(np.sign(r)) for r in CIRC_4[0]]
assert H_SIGNS_4 == ROW0_SIGNS_4                           # two mistakes, one value: one message
SCALED_4 = [EPS_4 * int(r) for r in ROW_4]                 # ε r: out of the cube
L2_4 = float(EPS_4) * float(np.linalg.norm(ROW_4))
MAX_H_4 = float(EPS_4 * int(np.abs(H_4).max()))           # the largest weight only
SIGNED_SUM_4 = float(EPS_4 * int(H_4.sum()))               # the weights with their signs

# CP5.5: a drug between blood and tissues, x_(t+1) = A x_t + B u_t, y_t = C x_t
A_5, B_5, C_5 = fmat([["0.5", "0.1"], ["0.3", "0.7"]]), [1, 0], [0, 1]
X0_5, U_5 = [Fraction(20), Fraction(4)], [6, 0, 2]


def run_5(a=A_5, inputs=U_5, steps=3):
    """The states x_0, ..., x_steps (an input missing from the list is 0)."""
    xs = [X0_5]
    for t in range(steps):
        u = inputs[t] if t < len(inputs) else 0
        xs.append([v + b * u for v, b in zip(fmatvec(a, xs[-1]), B_5)])
    return xs


XS_5 = run_5()
XS_T_5 = run_5(ftranspose(A_5))                            # A transposed
XS_NO_U0_5 = run_5(inputs=[0] + U_5[1:])
XS_LATE_5 = run_5(inputs=U_5[1:])                          # u_(t+1) used for x_(t+1): one step late
LAM_5 = eig2(A_5)                                          # 2/5 and 4/5
VEC_5 = {lam: [1, (lam - A_5[0][0]) / A_5[0][1]] for lam in LAM_5}   # (1, v): the first row of (A − λI) v = 0
assert all(fmatvec(A_5, v) == [lam * c for c in v] for lam, v in VEC_5.items())
assert np.allclose(sorted(np.linalg.eigvals(np.array(floats(A_5))).real), floats(LAM_5))
DIAG_5 = sorted([A_5[0][0], A_5[1][1]])                    # the diagonal coefficients
NEG_5 = sorted(-lam for lam in LAM_5)                      # the roots of λ² + tr λ + det
COEF_B_5 = np.linalg.solve(np.array([floats(VEC_5[lam]) for lam in LAM_5]).T, floats(B_5))   # B on the eigenvectors
H_5 = [sum(c * v for c, v in zip(C_5, fmatvec(fpow(A_5, k), B_5))) for k in range(20)]
assert np.allclose(floats(H_5), [sum(b * float(lam) ** k * float(VEC_5[lam][1]) for b, lam in zip(COEF_B_5, LAM_5))
                                 for k in range(20)])      # h_k = Σ (coefficient of B) λ^k (C v)
H12_5 = [fpow(A_5, k)[0][1] for k in range(20)]           # the coefficient (1, 2) of A^k
FIRST_5 = [fmatvec(fpow(A_5, k), B_5)[0] for k in range(20)]          # C = (1 0): the other coordinate
L1_5, L2_5 = LAM_5                                         # λ1 < λ2
BETA_5 = (H_5[1] - L2_5 * H_5[0]) / (L1_5 - L2_5)           # α + β = h_0 and α λ2 + β λ1 = h_1
ALPHA_5 = H_5[0] - BETA_5
assert all(ALPHA_5 * L2_5 ** k + BETA_5 * L1_5 ** k == H_5[k] for k in range(20))
LATE_5 = [ALPHA_5 * L2_5, BETA_5 * L1_5]                    # the formula started at k = 1
B_ON_5 = [Fraction(COEF_B_5[1]).limit_denominator(100), Fraction(COEF_B_5[0]).limit_denominator(100)]
assert B_ON_5[0] * VEC_5[L2_5][1] == ALPHA_5 and B_ON_5[1] * VEC_5[L1_5][1] == BETA_5   # times what C reads
FORCED_5 = sum(H_5[2 - k] * U_5[k] for k in range(3))      # y_3 = (h * u)_2 + C A^3 x_0
NO_SHIFT_5 = sum(H_5[3 - k] * U_5[k] for k in range(3))    # y_3 read as (h * u)_3: the shift forgotten
FREE_Y3_5 = sum(c * v for c, v in zip(C_5, fmatvec(fpow(A_5, 3), X0_5)))
assert FORCED_5 + FREE_Y3_5 == XS_5[3][1]
CARRIED_5 = H_5[3] * U_5[0]                                # u_0 carried by A^3 instead of A^2
I_MINUS_5 = [[int(i == j) - A_5[i][j] for j in range(2)] for i in range(2)]
EQ_5 = fmatvec(finv2(I_MINUS_5), [4 * b for b in B_5])
assert fmatvec(A_5, EQ_5) == [x - 4 * b for x, b in zip(EQ_5, B_5)]
EQ_T_5 = fmatvec(finv2(ftranspose(I_MINUS_5)), [4 * b for b in B_5])
COLUMNS_5 = [sum(col) for col in zip(*A_5)]
assert len(set(COLUMNS_5)) == 1                            # (1, 1) A = c (1, 1): the total is multiplied by c
FACTOR_5 = COLUMNS_5[0]
TOTAL_EQ_5 = 4 / (1 - FACTOR_5)                            # the total at equilibrium

# CP5.6: M, of trace 3/2 and determinant 3/4
M_6 = np.array([[0, -1], [0.75, 1.5]])
MF_6 = fmat([["0", "-1"], ["0.75", "1.5"]])
EIG_6 = np.linalg.eigvals(M_6)
LAM_6 = complex(EIG_6[np.argmax(EIG_6.imag)])
RHO_6 = float(np.abs(EIG_6).max())
assert abs(RHO_6 ** 2 - np.linalg.det(M_6)) < 1e-12        # two conjugate eigenvalues: |λ|² = λ λ̄ = det
THETA_6 = float(np.angle(LAM_6))
DET_6 = MF_6[0][0] * MF_6[1][1] - MF_6[0][1] * MF_6[1][0]
TR_6 = MF_6[0][0] + MF_6[1][1]
assert abs(LAM_6.real - float(TR_6 / 2)) < 1e-12
M2_6, M3_6, M6_6 = fpow(MF_6, 2), fpow(MF_6, 3), fpow(MF_6, 6)
assert M6_6 == [[-DET_6 ** 3, 0], [0, -DET_6 ** 3]]        # six steps of π/6: a half-turn, and a factor |λ|⁶
assert fpow(MF_6, 12) == [[DET_6 ** 6, 0], [0, DET_6 ** 6]]   # ((2/√3) M)^12 = (4/3)^6 M^12 = I: bounded powers
CUBED_6 = [[x ** 3 for x in row] for row in MF_6]          # each coefficient cubed
N1_6, NINF_6, N2_6 = (float(np.linalg.norm(M_6, o)) for o in (1, np.inf, 2))
assert N1_6 == float(norm_1(MF_6)) and NINF_6 == float(norm_inf(MF_6))
SIGNED_6 = [max(float(sum(c)) for c in zip(*MF_6)), max(float(sum(r)) for r in MF_6)]   # the signs kept
SMALLEST_6 = [min(float(sum(abs(x) for x in c)) for c in zip(*MF_6)), min(float(sum(abs(x) for x in r)) for r in MF_6)]
SV_6 = np.linalg.svd(M_6, compute_uv=False)
assert abs(SV_6[0] - N2_6) < 1e-12
GRAM_MAX_6 = float(np.linalg.eigvalsh(M_6.T @ M_6)[-1])
FROB_6 = float(np.linalg.norm(M_6))
SCALE_6 = 2 / math.sqrt(3)
CODES_6 = [behaviour(M_6), behaviour(SCALE_6 * M_6)]

# CP5.7 (1): f(x, y) = (0.5 x + y², x − 0.5 y) from p_0 = (2, 1)
HALF_7 = Fraction(1, 2)


def f_7(p):
    x, y = p
    return [HALF_7 * x + y * y, x - HALF_7 * y]


def jac_7(p):
    """One row per output: [[∂f1/∂x, ∂f1/∂y], [∂f2/∂x, ∂f2/∂y]]."""
    _, y = p
    return [[HALF_7, 2 * y], [Fraction(1), -HALF_7]]


PS_7 = [[Fraction(2), Fraction(1)]]
for _ in range(2):
    PS_7.append(f_7(PS_7[-1]))
J0_7, J1_7, J2_7 = (jac_7(p) for p in PS_7)
D2_7 = fmatmul(J1_7, J0_7)                                 # the jacobian of the last step on the left
STEP_7 = 1e-6
P0_7 = np.array(floats(PS_7[0]))
FD_7 = np.column_stack([(np.array(f_7(f_7(P0_7 + e))) - np.array(f_7(f_7(P0_7 - e)))) / (2 * STEP_7)
                        for e in STEP_7 * np.eye(2)])      # column j: the derivative along the input j
assert np.allclose(FD_7, floats(D2_7), atol=1e-6)          # finite differences on f∘f
NORMS_7 = [float(norm_inf(J1_7) * norm_inf(J0_7)), float(norm_inf(D2_7))]
SLIPS_C_7 = {"columns": [float(norm_1(J1_7) * norm_1(J0_7)), float(norm_1(D2_7))],
             "order": [float(norm_inf(J1_7) * norm_inf(J0_7)), float(norm_inf(fmatmul(J0_7, J1_7)))]}
assert SLIPS_C_7["columns"] == SLIPS_C_7["order"]          # two mistakes, one value: one message

# CP5.7 (2): the tank x_(t+1) = x_t − β √x_t + 1, x_0 = 36, β = 2
X0_7, BETA_7, DBETA_7 = 36, 2, Fraction("0.1")


def tank(beta, steps=3):
    xs = [float(X0_7)]
    for _ in range(steps):
        xs.append(xs[-1] - beta * math.sqrt(xs[-1]) + 1)
    return xs


XT_7 = [int(x) for x in tank(BETA_7)]
ROOTS_7 = [math.isqrt(x) for x in XT_7]
assert all(r * r == x for r, x in zip(ROOTS_7, XT_7))      # 36, 25, 16, 9: exact square roots
JT_7 = [1 - Fraction(BETA_7, 2 * r) for r in ROOTS_7[:3]]  # ∂f/∂x = 1 − β/(2√x)
BT_7 = [Fraction(-r) for r in ROOTS_7[:3]]                 # ∂f/∂β = −√x
TERMS_7 = [math.prod(JT_7[k + 1:3], start=Fraction(1)) * BT_7[k] for k in range(3)]
S3_7 = sum(TERMS_7)
s_t = Fraction(0)
for J_t, b_t in zip(JT_7, BT_7):
    s_t = J_t * s_t + b_t                                  # s_(t+1) = J_t s_t + b_t, s_0 = 0
assert s_t == S3_7
assert abs((tank(BETA_7 + 1e-6)[3] - tank(BETA_7 - 1e-6)[3]) / 2e-6 - float(S3_7)) < 1e-6
OWN_7 = [math.prod(JT_7[k:3], start=Fraction(1)) * BT_7[k] for k in range(3)]   # its own jacobian included
DX3_7 = DBETA_7 * S3_7
EXACT_7 = tank(BETA_7 + float(DBETA_7))[3] - tank(BETA_7)[3]

# CP5.10: the recursive filter, A = [[0.6, −0.5], [0.4, 0.6]], B = (1, 0), C = (0 1), x_0 = 0
A_10, B_10, C_10 = fmat([["0.6", "-0.5"], ["0.4", "0.6"]]), [1, 0], [0, 1]


def impulse(a, b, c, n):
    """h_k = C A^k B for k = 0, ..., n − 1, one product per step."""
    out, v = [], [Fraction(x) for x in b]
    for _ in range(n):
        out.append(sum(ci * vi for ci, vi in zip(c, v)))
        v = fmatvec(a, v)
    return out


H_10 = impulse(A_10, B_10, C_10, 80)
H12_10 = [fpow(A_10, k)[0][1] for k in range(5)]
FIRST_10 = impulse(A_10, B_10, [1, 0], 5)                  # C = (1 0): the other coordinate
U_10 = [4, -2, 0, 3]
XS_10 = [[Fraction(0), Fraction(0)]]
for t in range(7):
    u = U_10[t] if t < len(U_10) else 0
    XS_10.append([v + b * u for v, b in zip(fmatvec(A_10, XS_10[-1]), B_10)])
OUT_10 = [sum(c * v for c, v in zip(C_10, x)) for x in XS_10]           # y_t = C x_t
CONV_10 = np.convolve(floats(H_10[:8]), U_10)              # (h * u)_t
assert np.allclose(floats(OUT_10[1:7]), CONV_10[:6])       # y_(t+1) = (h * u)_t
RHO_10 = rho(A_10)
DET_10 = A_10[0][0] * A_10[1][1] - A_10[0][1] * A_10[1][0]
TR_10 = A_10[0][0] + A_10[1][1]
assert TR_10 ** 2 < 4 * DET_10 and abs(RHO_10 ** 2 - float(DET_10)) < 1e-12
GRAM_10 = fmatmul(ftranspose(A_10), A_10)
NORM2_10 = fsqrt(max(eig2(GRAM_10)))                       # the largest singular value, exact
NORMS_10 = [float(norm_1(A_10)), float(norm_inf(A_10)), float(NORM2_10)]
assert np.allclose(NORMS_10, [np.linalg.norm(np.array(floats(A_10)), o) for o in (1, np.inf, 2)])
SIGNED_10 = [max(float(sum(c)) for c in zip(*A_10)), max(float(sum(r)) for r in A_10)]
SMALLEST_10 = [min(float(norm_1([[x] for x in c])) for c in zip(*A_10)), min(float(norm_inf([r])) for r in A_10)]
assert SIGNED_10 == SMALLEST_10                            # two mistakes, one value: one message
FROB_10 = float(np.linalg.norm(np.array(floats(A_10))))
TARGET_10 = Fraction(1, 100)
K_10 = next(K for K in range(1, 500) if NORM2_10 ** K / (1 - NORM2_10) <= TARGET_10)
assert sum(abs(h) for h in H_10[K_10:]) <= TARGET_10       # the true tail is under the bound
NO_FACTOR_10 = next(K for K in range(1, 500) if NORM2_10 ** K <= TARGET_10)
WITH_RHO_10 = next(K for K in range(1, 500) if RHO_10 ** K / (1 - RHO_10) <= 0.01)
assert NO_FACTOR_10 == WITH_RHO_10                         # two mistakes, one value: one message
RHO_ONLY_10 = next(K for K in range(1, 500) if RHO_10 ** K <= 0.01)
G5_10 = H_10[:5]                                           # K = 5: h_0, ..., h_4
WORST_10 = sum(abs(h) for h in G5_10)
assert max(abs(sum(g * u for g, u in zip(G5_10, signs))) for signs in product((-1, 1), repeat=5)) == WORST_10
WORST_SLIPS_10 = {"four": sum(abs(h) for h in H_10[:4]), "six": sum(abs(h) for h in H_10[:6])}
I_MINUS_10 = [[int(i == j) - A_10[i][j] for j in range(2)] for i in range(2)]
LIMIT_10 = sum(c * v for c, v in zip(C_10, fmatvec(finv2(I_MINUS_10), B_10)))
assert abs(float(sum(H_10)) - float(LIMIT_10)) < 1e-8      # also the sum of the impulse response
LIMIT_T_10 = sum(c * v for c, v in zip(C_10, fmatvec(finv2(ftranspose(I_MINUS_10)), B_10)))
PARTIAL_10 = sum(H_10[:5])

**CP5.1 — Convolution de deux suites, corrélation croisée et matrice de Toeplitz**

In [3]:
wbm.record("CP5.1a", ints(AB_1), mistakes={"la convolution retourne l'un des deux facteurs : dans (a * b)ₘ = Σ aᵢ bₘ₋ᵢ, la somme des deux indices vaut m": ints(SLID_1), "la liste va du terme constant au terme de plus haut degré": ints(AB_1[::-1])})
wbm.record("CP5.1b", ALT_1, mistakes={"une somme alternée change le signe d'un terme sur deux : relis sa définition": SUM_1, "vérifie le signe du premier terme, (−1)⁰ = 1 ; et la valeur d'un produit de polynômes est le produit de leurs valeurs, pas leur somme": {-ALT_1, P_1 + Q_1}})
wbm.record("CP5.1c", ints(CORR_1), mistakes={"la corrélation croisée ne retourne pas le noyau : la fenêtre i calcule Σᵤ wᵤ x_(i+u)": ints(VALID_1)})
wbm.record("CP5.1d", ints(FULL_1), mistakes={"la convolution retourne le noyau : (x * w)ₘ = Σᵢ xᵢ w_(m−i)": ints(FULLCORR_1), "le mode plein garde toutes les positions où le noyau touche le signal, pas seulement les fenêtres complètes": ints(VALID_1)})
wbm.record("CP5.1e", ints(SAME_1), mistakes={"le mode \"same\" de NumPy ne commence pas au début de la convolution pleine : relis la règle de son indice de départ": ints(FULL_1[:N_1]), "pour un noyau de longueur paire, NumPy enlève une valeur de moins au début qu'à la fin": ints(FULL_1[START_1 + 1:START_1 + 1 + N_1])})
wbm.record("CP5.1f", ints(T3_1), mistakes={"Tᵢⱼ = w_(i−j) : dans une colonne, l'indice du noyau croît avec l'indice de ligne": ints(toeplitz(W_1[::-1], 3)), "la matrice a une ligne par valeur de la sortie et une colonne par valeur de l'entrée": ints(T3_1.T)})
wbm.record("CP5.1g", NNZ_1, mistakes={"un coefficient nul du noyau ne donne aucun coefficient non nul dans la matrice": K_1 * 40, "la question compte les coefficients non nuls, pas tous les coefficients de la matrice": int(T40_1.size)})

WBM_ANSWER {"decimals": 0, "element_hashes": ["eb784f1a1b2872fb4b7c6a9d515841211763fd110259080b6cc54cafa909efab", "4ad497ea3efb68e7225b47bd8bcac8a99c382d03b52546df9d85814ae42c18d0", "2e2a0a59eef089b30d49ee20312f519be4ec22cb3370294af2f17ab433c0cb63", "ba7870e2af3a573a3bf1c17b1f4930645f7f2e39f367b632ddb03958d1329cc5", "cddd680dafed7a7c1784e0b8d44fad935b99bd5e2cb4ac4b3eb1e05acad77ba3", "4d1669e4c575fc2c3cac8206e6ddcaa2edb599302e2aa6d275bd79e80bfcff5f"], "hash": "41aaebdde7d368098efe15f0ffe520ddb915d25c2a44b01ebabc311308f1a10a", "id": "CP5.1a", "integer": true, "kind": "array", "mistakes": {"481b8650e2e9a564ea186dacc34c8802771b64f70a12f50421d2c7c90c396993": "la liste va du terme constant au terme de plus haut degré", "e71cfbd9e7e4202b5e6bb92e420d3d4acb48cfe9959f6a5bdebff2d94027ef3e": "la convolution retourne l'un des deux facteurs : dans (a * b)ₘ = Σ aᵢ bₘ₋ᵢ, la somme des deux indices vaut m"}, "shape": [6]}
📝 Ex CP5.1a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"hash": "ece129

**CP5.2 — Positions d'une fenêtre, pas, dilatation, et la convolution transposée d'un signal**

In [4]:
wbm.record("CP5.2a", POS_A_2, mistakes={"le nombre de positions est une partie entière plus 1 : la première position compte": POS_A_2 - 1, "la partie entière est inférieure : une fenêtre qui dépasse du signal n'est pas une position": CEIL_A_2})
wbm.record("CP5.2b", POS_B_2, mistakes={"avec une dilatation d, la fenêtre s'étend sur d(k − 1) + 1 cases": NODIL_B_2, "vérifie chaque terme de la formule : le padding est ajouté des deux côtés, l'étendue d'une fenêtre dilatée est d(k − 1) + 1, et le nombre est la partie entière plus 1": B_SLIPS_2, "la partie entière est inférieure, et la formule tient compte du padding et de la dilatation de la question": B_UP_2, "le padding p ajoute p zéros de chaque côté du signal avant de compter les positions": NOPAD_B_2})
wbm.record("CP5.2c", WINDOWS_2, ordered=False, mistakes={"le nombre de positions est la partie entière plus 1": NO_ONE_2, "une position existe quand la dernière case de sa fenêtre, si + k − 1, reste dans le signal : vérifie le − 1 de la formule": NO_MINUS_2, "la question demande la liste de toutes les longueurs, pas seulement les deux bornes": [WINDOWS_2[0], WINDOWS_2[-1]], "la partie entière confond plusieurs longueurs : donne-les toutes": [WINDOWS_2[0]], "plusieurs longueurs donnent ce nombre de positions : la question les demande toutes": [WINDOWS_2[-1]]})
wbm.record("CP5.2d", SAME_2, mistakes={"TensorFlow et Keras mettent le zéro en trop à la fin": [OUT_2, SAME_2[2], SAME_2[1]], "avec un pas, le padding « même taille » ne vaut pas k − 1 zéros : il en faut juste assez pour que la dernière fenêtre tienne": [OUT_2, (K_2 - 1) // 2, (K_2 - 1) // 2], "un nombre de zéros est entier : quand le total est impair, un côté en reçoit un de plus": [OUT_2, TOTAL_2 / 2, TOTAL_2 / 2]})
wbm.record("CP5.2e", ints(TRANS_2), mistakes={"chaque entrée yᵢ dépose le tampon yᵢ w, le noyau dans l'ordre, à partir de la case s·i": ints(REVERSED_2), "là où deux tampons se recouvrent, leurs valeurs s'additionnent": ints(OVERWRITTEN_2)})
wbm.record("CP5.2f", SIZE_2, mistakes={"le padding de sortie ajoute p_out valeurs à la fin": SIZE_SLIPS_2["no p_out"], "dans une convolution transposée, le padding p retire des valeurs au lieu d'en ajouter": SIZE_SLIPS_2["p added"], "le padding p retire p valeurs à chaque bout": SIZE_SLIPS_2["p once"], "le padding change aussi la taille d'une convolution transposée": SIZE_SLIPS_2["p ignored"]})
wbm.record("CP5.2g", COUNTS_2, ordered=False, mistakes={"quand s ne divise pas k, les cases intérieures ne reçoivent pas toutes le même nombre de tampons : donne toutes les valeurs": [COUNTS_2[0]], "les cases intérieures ne reçoivent pas toutes le même nombre de tampons quand s ne divise pas k : donne chacune des valeurs": [COUNTS_2[-1]]})

WBM_ANSWER {"hash": "8ac97a37938a4ef0d5448e97deaa55ad8b08ac06106ac52544b5eb97482a4ce6", "id": "CP5.2a", "kind": "int", "magnitude": 2, "mistakes": {"6a4425154887f928c91d98a3132d5e8264693ee028f5eeeb5b04395b9d4d766b": "le nombre de positions est une partie entière plus 1 : la première position compte", "8ed5b731b7d453005d9a29d61d7b1b823de2d3678266f30d8fbc2e32e97ef8fe": "la partie entière est inférieure : une fenêtre qui dépasse du signal n'est pas une position"}, "sign": 1}
📝 Ex CP5.2a : réponse enregistrée (int).
WBM_ANSWER {"hash": "9ae1e63989f51f38966ed49a628d850472d6bfedb9effd7db97184de170b1369", "id": "CP5.2b", "kind": "int", "magnitude": 2, "mistakes": {"3390e76ebf58e66414dae1975386a2fcab19d2b496012be67c038a4f471d320d": "la partie entière est inférieure, et la formule tient compte du padding et de la dilatation de la question", "bc8205b2923fac65b140bb367065992d73a0a89592819b27d4a06bce555d9a41": "vérifie chaque terme de la formule : le padding est ajouté des deux côtés, l'étendue d'

**CP5.3 — Une grille filtrée par un noyau séparable : paramètres, coût et champ récepteur**

In [5]:
wbm.record("CP5.3a", [ints(A_3), ints(B_3)], mistakes={"K_(u,v) = a_u b_v : a donne les lignes de K, b ses colonnes": [ints(B_3), ints(A_3)], "la normalisation demandée porte sur b : b₀ = 1": floats([A_3 / A_3[0], B_3 * A_3[0]])})
wbm.record("CP5.3b", ints(Y_3), mistakes={"une corrélation croisée ne retourne pas le noyau : Y_(i,j) = Σ K_(u,v) G_(i+u, j+v)": ints(CONV_3), "u est l'indice de ligne : K_(u,v) multiplie la case située u lignes plus bas et v colonnes plus à droite": ints(TRANSPOSED_3)})
wbm.record("CP5.3c", H_FIRST_3, mistakes={"la question demande le calcul qui commence par la passe horizontale, sur chaque ligne de G": V_FIRST_3, "la première passe doit fournir à la seconde toutes les valeurs qu'elle lit, pas seulement celles des lignes de la sortie": NARROW_3, "compte toutes les multiplications, même par 1 ou par −1": NOT_ONE_3})
wbm.record("CP5.3d", SHAPE_3, mistakes={"avec un pas, la sortie a moins de lignes et de colonnes : compte ses positions": SHAPE_SLIPS_3["stride"], "le padding (1 en hauteur, 2 en largeur) s'ajoute de chaque côté avant de compter les positions": SHAPE_SLIPS_3["padding"], "la sortie a un canal par filtre": SHAPE_SLIPS_3["channels"], "le padding s'ajoute des deux côtés de chaque axe : en haut et en bas, à gauche et à droite": SHAPE_SLIPS_3["one side"], "le padding vaut 1 en hauteur (le long des lignes) et 2 en largeur (le long des colonnes)": SHAPE_SLIPS_3["padding swapped"], "un noyau 3 × 5 a 3 lignes et 5 colonnes : 3 en hauteur, 5 en largeur": SHAPE_SLIPS_3["kernel swapped"], "la forme demandée est (C, H, W) : les canaux d'abord": SHAPE_SLIPS_3["order"]})
wbm.record("CP5.3e", PARAMS_3, mistakes={"chaque canal de sortie a un biais": PARAMS_3 - C_OUT_3, "un biais par canal de sortie, pas par canal d'entrée": PARAMS_3 - C_OUT_3 + C_IN_3})
wbm.record("CP5.3f", MACS_3, mistakes={"avec un pas, la sortie a moins de lignes et de colonnes : compte ses positions": NOSTRIDE_3, "le padding s'ajoute de chaque côté avant de compter les positions de sortie": NOPAD_3, "chaque noyau de sortie a C_in × k_h × k_w poids, un par canal d'entrée": NO_CIN_3, "un biais s'ajoute sans multiplication : le coût compte une multiplication-addition par poids": WITH_BIAS_3})
wbm.record("CP5.3g", SEP_3, mistakes={"chacune des deux convolutions a un biais par canal de sortie": NO_BIAS_3})
wbm.record("CP5.3h", [RF_3, STRIDE_3], mistakes={"un étage dilaté ajoute d(k − 1) fois le produit des pas précédents, pas k − 1 fois": [field(STACK_3, dilation=False), STRIDE_3], "chaque étage ajoute d(k − 1) cases de son entrée, et une case de son entrée vaut le produit des pas précédents en cases du signal d'origine": [NOJUMP_NODIL_3, STRIDE_3], "une case de l'entrée d'un étage vaut le produit des pas précédents en cases du signal d'origine": [NOJUMP_3, STRIDE_3], "additionner les tailles des noyaux oublie les pas : les pas des étages précédents multiplient ce qu'ajoute chaque étage": [KSUM_3, STRIDE_3], "additionner les étendues des fenêtres oublie les pas, et chaque étage n'ajoute que d(k − 1) cases de son entrée": [SPANS_3, STRIDE_3], "le pas total est le produit des pas, pas leur somme": [RF_3, sum(s for _, s, _ in STACK_3)], "un pas ne change pas la contribution de son propre étage : seuls les pas des étages précédents la multiplient": [field(STACK_3, own_stride=True), STRIDE_3]})

WBM_ANSWER {"decimals": 0, "element_hashes": ["3c12e58cd82611226c02bdad8601ea6fe6dac654cf665f1a565c2f526fa03c15", "5b18ac634a4a999b67abe4bfd9e37fb0945eba66265f1db654058e6e40db82fb", "eb14603c376565cff3534fd50e536b7e9275351fd2db26991759f5600b89aa54", "cf103b00be03a5785789f4a61065ff13870bb2170005ad21d7c4bd5156f13ada", "51299c07b7cffcef42501bb8af8745b284be6124060ce69a7c4dec923bc776f9", "5f2bdf3a1afd8c265bcef0f4c738903abc1f1a8fa42b81bfb4acee5c57ef6e80"], "hash": "3cfc0dcb896ed5cf94cced0d5bb0e1efddc1b417562458bbf8161ff4f9112353", "id": "CP5.3a", "integer": true, "kind": "array", "mistake_decimals": 2, "mistakes": {"312a523d4ee8692461908cef60d9d3033096957c552bbeef764421f347c8891c": "K_(u,v) = a_u b_v : a donne les lignes de K, b ses colonnes", "7f761702bd99fffebc9f20e5eb26454d52d37777a5e2ced4444f132d56928975": "la normalisation demandée porte sur b : b₀ = 1"}, "shape": [2, 3]}
📝 Ex CP5.3a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["d01cabac38f268

**CP5.4 — Équivariance, matrice circulante et le pire cas d'une perturbation bornée**

In [6]:
wbm.record("CP5.4a", ints(Z_4), mistakes={"dans une convolution circulaire, les indices se lisent modulo n : les termes qui dépassent n − 1 reviennent au début": ints(UNFOLDED_4), "la convolution circulaire retourne aussi le noyau : (x ⊛ h)ₘ = Σᵢ xᵢ h_((m−i) mod n)": ints(CIRC_CORR_4), "l'indice de h est (m − i) mod n, pas (m + i) mod n : la convolution retourne le noyau": ints(PLUS_4)})
wbm.record("CP5.4b", ints(CIRC_4[0]), mistakes={"h est la première colonne de circ(h) ; la question demande la première ligne": ints(H_4), "le coefficient (0, j) vaut h_((0−j) mod n) : le retourné circulaire n'est pas le simple renversement": ints(H_4[::-1])})
wbm.record("CP5.4c", ints(ZP2_4), mistakes={"décaler l'entrée d'une convolution circulaire laisse-t-il sa sortie à sa place ?": ints(Z_4), "(Px)ᵢ = x_((i−1) mod n) : P pousse les valeurs vers les indices croissants": ints(BACK_4), "P² décale deux fois": ints(ONCE_4)})
wbm.record("CP5.4d", floats(DELTA_4), decimals=6, mistakes={"le changement de z₁ est le produit scalaire de δ avec la ligne 1 de circ(h) (les lignes sont numérotées à partir de 0), pas avec h": floats(H_SIGNS_4), "on cherche le changement le plus grand vers le haut : z₁ doit augmenter": floats([-d for d in DELTA_4]), "chaque |δᵢ| doit rester au plus 0,2 : ce δ sort des tolérances": floats(SCALED_4)})
wbm.record("CP5.4e", float(WORST_INF_4), decimals=6, mistakes={"chaque δᵢ se choisit indépendamment des autres : tous les poids comptent, pas seulement le plus grand": MAX_H_4, "aucun terme ne doit en compenser un autre : le signe de chaque δᵢ se choisit séparément": SIGNED_SUM_4, "ici chaque coordonnée de δ est bornée séparément (|δᵢ| ≤ 0,2) : ce n'est pas la boule L2": {L2_4, round(L2_4, 3), round(L2_4, 4)}, "une augmentation est positive : δ = 0 donne déjà un changement nul": -float(WORST_INF_4)})
wbm.record("CP5.4f", WORST_TWO_4, decimals=3, mistakes={"la contrainte porte cette fois sur ‖δ‖₂ : δ ne peut plus prendre ±0,2 sur chaque coordonnée à la fois": float(WORST_INF_4), "la norme L2 est la racine carrée de la somme des carrés": float(EPS_4) * float(ROW_4 @ ROW_4), "sur une boule L2, tous les poids comptent, pas seulement le plus grand": float(EPS_4) * float(np.abs(ROW_4).max()), "le rayon de la boule est ε = 0,2, pas 1 : le pire cas est proportionnel à ε": float(np.linalg.norm(ROW_4))})

WBM_ANSWER {"decimals": 0, "element_hashes": ["76dc2e38c1f297a4df2f511d20f2e724e01f3c85a1af4075d18d3317f7bd93c5", "38bb097af6f7b5dd240963414be28bd18e7d6e2474458d66ae1cb2abcfffb12a", "240476c35233b9ec4b784457764d5d3e740a428cfe7862522fcdf413c6b1182c", "72ce73c3e1f2836f3517729036a2119a1a3883a54c8b204806c2778e259e31f6", "6d152b8470a6b0205c9eae8edd16507fbaf515b32c658f4f4d19b090770ca9c3"], "hash": "b04cf577d2db944a463098714badcf4fa358238a664f0df252dff38aec31fe5f", "id": "CP5.4a", "integer": true, "kind": "array", "mistakes": {"2ef70a750e54ce08e017d0a5d278f01cf41f0f28078a2459cd62db5293893883": "la convolution circulaire retourne aussi le noyau : (x ⊛ h)ₘ = Σᵢ xᵢ h_((m−i) mod n)", "5d548d3d6c48e3b6cdb5035ce68c37d6fa2d20d9fe2131149d986420b4e87eab": "l'indice de h est (m − i) mod n, pas (m + i) mod n : la convolution retourne le noyau", "a1c80e5410bb92d6b43e4de5a1fde7643e1518762f4db9ed72f2dc4cfd92bb22": "dans une convolution circulaire, les indices se lisent modulo n : les termes qui dépassent n

**CP5.5 — Une récurrence avec entrée : solution et réponse impulsionnelle**

In [7]:
wbm.record("CP5.5a", floats(XS_5[2]), decimals=6, mistakes={"en convention colonne, la ligne i de A donne la coordonnée i de A xₜ": floats(XS_T_5[2]), "la perfusion u₀ entre dans x₁ = A x₀ + B u₀": floats(XS_NO_U0_5[2]), "l'entrée uₜ sert à calculer xₜ₊₁, pas xₜ": floats(XS_LATE_5[2]), "la question demande x₂ : deux pas depuis x₀": floats(XS_5[1])})
wbm.record("CP5.5b", floats(XS_5[3]), decimals=6, mistakes={"chaque pas ajoute B uₜ, pour chaque entrée reçue": floats(fmatvec(A_5, XS_5[2])), "en convention colonne, la ligne i de A donne la coordonnée i de A xₜ": floats(XS_T_5[3]), "l'entrée uₜ sert à calculer xₜ₊₁, pas xₜ": floats(XS_LATE_5[3])})
wbm.record("CP5.5c", floats(LAM_5), ordered=False, decimals=6, mistakes={"les valeurs propres d'une matrice non triangulaire ne sont pas ses coefficients diagonaux : résous λ² − tr(A) λ + det(A) = 0": floats(DIAG_5), "vérifie les signes du polynôme caractéristique λ² − tr(A) λ + det(A)": floats(NEG_5)})
wbm.record("CP5.5d", floats(H_5[:4]), decimals=6, mistakes={"la réponse impulsionnelle commence à h₀ = C B": floats(H_5[1:5]), "dans hₖ = C Aᵏ B, B choisit une colonne de Aᵏ et C une ligne : vérifie lesquelles": floats(H12_5[:4]), "hₖ est une sortie : relis quelle coordonnée de l'état C lit": floats(FIRST_5[:4])})
wbm.record("CP5.5e", floats([ALPHA_5, BETA_5]), decimals=6, mistakes={"α multiplie les puissances de la plus grande valeur propre, λ₂": floats([BETA_5, ALPHA_5]), "la formule vaut dès k = 0 : écris-la pour h₀ et pour h₁": floats(LATE_5), "hₖ = C Aᵏ B : les coefficients de B sur les vecteurs propres ne suffisent pas, C lit encore une coordonnée de chaque vecteur propre": floats(B_ON_5)})
wbm.record("CP5.5f", float(FORCED_5), decimals=6, mistakes={"la part due à la perfusion est la réponse forcée ; la réponse libre vient de x₀": float(FREE_Y3_5), "y₃ contient aussi la réponse libre : la question n'en demande qu'une part": float(XS_5[3][1]), "l'entrée uₖ est portée par A^(t−1−k) jusqu'à xₜ": float(CARRIED_5), "y₃ = C x₃, et y_(t+1) = (h * u)_t : un pas de décalage entre la sortie et la convolution": float(NO_SHIFT_5)})
wbm.record("CP5.5g", floats(EQ_5), decimals=6, mistakes={"l'équilibre vérifie x⋆ = A x⋆ + B u : la matrice du système est I − A, pas sa transposée": floats(EQ_T_5), "l'équilibre vérifie x⋆ = A x⋆ + B u, et la matrice du système est I − A, pas sa transposée": [float(EQ_T_5[0]), round(float(EQ_T_5[1]), 4)], "ta réponse donne-t-elle à chaque compartiment la quantité totale ? Résous (I − A) x⋆ = B u coordonnée par coordonnée": floats([TOTAL_EQ_5, TOTAL_EQ_5])})
wbm.record("CP5.5h", float(FACTOR_5), decimals=6, mistakes={"ce facteur vaut pour tout xₜ : lis-le sur les sommes des colonnes de A": float(min(LAM_5)), "la trace est la somme des valeurs propres, pas le facteur appliqué à la quantité totale": float(A_5[0][0] + A_5[1][1]), "le facteur multiplie la quantité qui reste : la part éliminée n'en est que le complément": float(1 - FACTOR_5), "le médicament éliminé par chaque compartiment quitte le système : la quantité totale ne se conserve pas": 1})

WBM_ANSWER {"array_rounded": {"1": "5d97cd0341bbeeb16542e29a414564466e819c2c3f6405c5629106548bb852f2"}, "decimals": 6, "element_coarse": [["611707e32f47ddd1a488ef536beff7b806b6e85c4f5f7319e82959e790bd3065"], ["b34df1c232afad9bfd145ad3678dfde0783841977ccd075844329415e5f3a715"]], "element_coarse_any": [["1df1c8680ee9c2ddd2c0b51bccd3379add4fe22963fb2d3493b390010496e4ba", "a960feb3d8bbfbfc477c4116ebd958a581647d32e99eb1a1bbff231d9e7b3f74", "3529f21ccfea5442c6daedddb7eea9597fbf231cd0d08c7cc0b5ff9da4e46567", "2fd5eb1933fab0052e2eac54eddd29fcfa26fb8948be1252ccc04b706b1ac318", "1e9d492be8e290de04901ffb24faf271e74f0db475c467709e24338e5f9c6916"], ["d72b45deeab0ff9db9f9459b0e7a41201174d53e06df5831c6c1c9d524121845", "1a9a911abbccd39fbaa22bc8ce74a773ae6c3ef3721bd6255133ce491d3c4783", "744f3dd1b512ee09944d7fa8e2921e18cbf025dc48ed0961bee63b986acf73d4", "b0091b6ecbbab6b920034ddbe6f143f35c37f3a8598fe6e1dedd986aac6d6977", "cc61076d0bfe1a184fe43cf66db81c6dc31cda62720921362f36c6db75430a45"]], "element_hash

**CP5.6 — Valeurs propres complexes, rayon spectral, normes et croissance transitoire**

In [8]:
wbm.record("CP5.6a", [LAM_6.real, LAM_6.imag], decimals=4, mistakes={"la question demande la valeur propre de partie imaginaire positive": [LAM_6.real, -LAM_6.imag], "λ = (tr ± √Δ)/2 : vérifie la division par 2": [2 * LAM_6.real, 2 * LAM_6.imag]})
wbm.record("CP5.6b", RHO_6, decimals=4, mistakes={"le module de a + ib est √(a² + b²) : il tient compte de la partie imaginaire, et c'est une racine carrée": {float(TR_6 / 2), float(DET_6)}})
wbm.record("CP5.6c", as_pi(THETA_6), kind="exact", angle=True, mistakes={"l'argument d'une valeur propre de partie imaginaire positive est dans ]0, π[": as_pi(-THETA_6), "cos θ = Re(λ)/|λ| et sin θ = Im(λ)/|λ| : vérifie lequel tu as utilisé": as_pi(math.pi / 2 - THETA_6), "ton angle est celui de la valeur propre conjuguée : l'argument de λ, de partie imaginaire positive, est dans ]0, π[": as_pi(2 * math.pi - THETA_6), "le cosinus de l'argument a le signe de la partie réelle": as_pi(math.pi - THETA_6), "c'est le même angle à 2π près : donne l'argument principal, dans ]−π, π]": as_pi(THETA_6 + 2 * math.pi), "même angle, à 2π près : l'argument principal est celui qui tombe dans ]−π, π]": as_pi(THETA_6 - 2 * math.pi)})
wbm.record("CP5.6d", floats(M3_6), decimals=6, mistakes={"M³ est le produit matriciel M × M × M, pas le cube de chaque coefficient": floats(CUBED_6), "M³ a trois facteurs : multiplie encore une fois par M": floats(M2_6)})
wbm.record("CP5.6e", floats(M6_6), decimals=6, mistakes={"vérifie les signes : fais chaque produit ligne par colonne": floats([[-x for x in row] for row in M6_6])})
wbm.record("CP5.6f", [N1_6, NINF_6], decimals=6, mistakes={"les normes ‖·‖₁ et ‖·‖∞ additionnent les valeurs absolues des coefficients, pas les coefficients avec leurs signes": SIGNED_6, "‖·‖₁ prend les sommes de colonnes, ‖·‖∞ les sommes de lignes": [NINF_6, N1_6], "‖·‖₁ et ‖·‖∞ prennent la plus grande des sommes, colonne par colonne ou ligne par ligne": SMALLEST_6})
wbm.record("CP5.6g", N2_6, decimals=4, mistakes={"‖M‖₂ est une valeur singulière : la racine carrée d'une valeur propre de MᵀM": GRAM_MAX_6, "le nombre que tu donnes est-il le plus grand facteur d'allongement ‖M x‖/‖x‖ ? Essaie quelques vecteurs": RHO_6, "la norme de Frobenius n'est pas la norme induite ‖·‖₂": FROB_6, "‖M‖₂ est la PLUS GRANDE valeur singulière": float(SV_6[-1])})
wbm.record("CP5.6h", CODES_6, mistakes={"multiplier une matrice par 2/√3 multiplie ses valeurs propres par 2/√3 : regarde leur nouveau module": [CODES_6[0], 1], "que deviennent les puissances d'une matrice diagonalisable dont les valeurs propres ont un module égal à 1 ?": [CODES_6[0], 3], "Aᵗ → 0 si et seulement si ρ(A) < 1, même quand une norme de A dépasse 1": [2, CODES_6[1]], "le comportement des puissances se lit sur le rayon spectral, pas sur une norme": [2, 3]})

WBM_ANSWER {"decimals": 4, "element_coarse": [["f5f299bdb7126708701d66f5007306350582f91edcda9e10554aa065316e72eb"], ["0fb6262659ed5f3a121395bcebd18590e1c12d42b66c857711883563789f4d85"]], "element_coarse_any": [["f09e57040fd2641c5ff745b19013bf843508a4ec535657b2be76ddc97eb2d697", "1bdf9468b555fd8368d843f592526b4778fd072ca5523e51a0c2b2624997aa90", "81ed98dcc2c7d032a23954e334142081bf3be3fac1de6dc7219eabe4e9fbad8b"], ["2e5d785ada87bc160a4a6cbd85353fa3b115ba1280bba00921316f6b80e184a2", "e56593111263ba5fb25e3789ae6f24e0ff92584adda88d913b53008aac12d713"]], "element_hashes": ["62f544ee0e8636b9986f39ce4be4580e07d4ffe856a1c151c79ef8d7969db044", "e8d40eb88149667126784ae7d2a842b4278385bab294c207f49d0a6f4aa3036d"], "hash": "399d369200c0a2ee81d6a031c381dc95b4f187309fa27cb9c98f05982eed3900", "id": "CP5.6a", "kind": "array", "mistakes": {"412ba87622a03d9bcd3493b7a1bcf48258d8b4fc1ee3e266bdf0810e61276ee6": "λ = (tr ± √Δ)/2 : vérifie la division par 2", "6e81f773a4b19c4ce6ac1032c0e474ca3ca5401b2e3a91b78cb

WBM_ANSWER {"angle": true, "hash": "494150178026c0d05ce3d4ed8906e1a9e8541dd687fb6b0ffa33e06a519a603e", "id": "CP5.6c", "kind": "exact", "mistakes": {"08d5d013efca926dad55471581648fd0ba310c27d70523e5449c77d942759e2d": "l'argument d'une valeur propre de partie imaginaire positive est dans ]0, π[", "2434eab1d85860d96d8ac25db5e2dba56a5d789d06f05e59fe8d9042b2aec908": "c'est le même angle à 2π près : donne l'argument principal, dans ]−π, π]", "81c0d1630cc6bdb6ebdb9d16691a978d6603453b23927c88a9eb7264358b4787": "cos θ = Re(λ)/|λ| et sin θ = Im(λ)/|λ| : vérifie lequel tu as utilisé", "d39b2a83e23b334aa2e569cf1caac234f04c38af244e8fb94316b1c52c0a611a": "ton angle est celui de la valeur propre conjuguée : l'argument de λ, de partie imaginaire positive, est dans ]0, π[", "ea57e61fc557eef2d44d0dc491eb5ff53e1749fdfe781c3ca7db1603bf87bb0a": "le cosinus de l'argument a le signe de la partie réelle", "eff725e97a3fd70a4a1997b7243f55e6552374fd649ca143605c02cff8d729bf": "même angle, à 2π près : l'argument 

**CP5.7 — Produits de jacobiennes le long d'une trajectoire, et sensibilité à un paramètre utilisé à chaque pas**

In [9]:
wbm.record("CP5.7a", floats(PS_7[2]), decimals=6, mistakes={"la question demande p₂ : f s'applique deux fois": floats(PS_7[1])})
wbm.record("CP5.7b", floats(D2_7), decimals=6, mistakes={"un produit de jacobiennes se range le dernier pas à gauche": floats(fmatmul(J0_7, J1_7)), "chaque jacobienne s'évalue en son propre point : le second pas part de p₁": floats(fmatmul(J0_7, J0_7)), "la jacobienne du pas t s'évalue au point de départ de ce pas, pₜ": floats(fmatmul(J2_7, J1_7)), "une jacobienne a une ligne par coordonnée de sortie": floats(ftranspose(D2_7)), "le premier pas part de p₀ : sa jacobienne s'évalue en p₀": floats(fmatmul(J1_7, J1_7))})
wbm.record("CP5.7c", NORMS_7, decimals=6, mistakes={"‖·‖∞ prend la plus grande somme des valeurs absolues d'une ligne, sur le produit rangé le dernier pas à gauche": SLIPS_C_7["columns"], "l'ordre demandé : le produit des normes, puis la norme du produit": NORMS_7[::-1]})
wbm.record("CP5.7d", floats(TERMS_7), decimals=6, mistakes={"chaque effet de β est ensuite porté jusqu'à x₃ par les jacobiennes des pas suivants": floats(BT_7), "range les termes du pas 0 au pas 2": floats(TERMS_7[::-1]), "la jacobienne du pas où β sert ne porte pas son propre effet : seules celles des pas suivants le portent": floats(OWN_7)})
wbm.record("CP5.7e", float(S3_7), decimals=6, mistakes={"chaque effet de β est porté jusqu'à x₃ par les jacobiennes des pas suivants": float(sum(BT_7)), "s₃ additionne les effets de β à tous les pas où il sert": float(BT_7[-1]), "la jacobienne du pas où β sert ne porte pas son propre effet : seules celles des pas suivants le portent": float(sum(OWN_7))})
wbm.record("CP5.7f", float(DX3_7), decimals=6, mistakes={"ta réponse est-elle un changement de hauteur, en cm, pour la variation de β donnée ?": float(S3_7), "un orifice qui laisse sortir plus d'eau fait-il monter ou baisser le niveau ? Vérifie le signe": -float(DX3_7), "c'est la variation exacte ; la question demande l'approximation au premier ordre": {EXACT_7, *(round(EXACT_7, d) for d in (2, 3, 4, 5)), math.trunc(EXACT_7 * 1e4) / 1e4}})

WBM_ANSWER {"array_rounded": {"1": "23e764a72e125033082beb92613a9f38dfc4bbaaed3b980dcea1f2d66ba3687b"}, "decimals": 6, "element_coarse": [["8694bdb8e98a32c67a7ee20bff980add4989d9c4be760c75fd52c04f21490b88"], ["5593e3c08de4651dbeaac0707f7e080a921e0f7d727e39b15143459ecaf56abb"]], "element_coarse_any": [["746a36d3c7e01f88bc4d41cd9561cca365c562b10687851ea3ffca6ca89bcc56", "b4cd67500a25b55ccc504edafddc31bd312d572c2f9ad9f943f13f0cdae9cc68", "713ce12c1c4506153024cfb5ea56228db9db338dd721b1e5fb016d6bf79c505b", "f15c1cb5e72de6e66abd44cd3fe6ffc068aef44c04ddb3a24b28a4258124c33b", "f10386ded0b5e17146234c0d035293a26121755b8810665ae4685bb5bfe12ae4", "2bd1aa9eff3d83b2e07cecb2ae4878c464206137d95953ddbc7a3b6b6379bda0"], ["0955d48eaee98707743577582b849b9c4b8213a09d5249dafcebae69e34ed223", "561bdaed35b1e6337ea180920882829bb8d817bba597ab7cf13e6ec911b18361", "ddf985a0549aff934964acd169516166374f5b02399d57d56f2a05201e05727a", "71af1d1eaaaf72e69ffe89c1aeb9680ce78353cf99c76a4f2347e7e72616f740", "2a2507128b45a2

**CP5.9 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [10]:
wbm.record("CP5.9a", "ABDEG", mistakes={"il manque une des lignes fausses : il y en a exactement cinq": {'ABDE', 'ABDG', 'ABEG', 'ADEG', 'BDEG'}, "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise": {'ABCDEG', 'ABDEFG', 'ABDEGH'}, "une ligne juste est citée à la place d'une ligne fausse": {'ABCDE', 'ABCDG', 'ABCEG', 'ABDEF', 'ABDEH', 'ABDFG', 'ABDGH', 'ABEFG', 'ABEGH', 'ACDEG', 'ADEFG', 'ADEGH', 'BCDEG', 'BDEFG', 'BDEGH'}, "ce sont les trois lignes justes ; la question demande les cinq lignes fausses": 'CFH'})

WBM_ANSWER {"hash": "409ca39eb59ac1ce06f22b57173c9da1bf8c41430d4abe0117a42def28a22891", "id": "CP5.9a", "kind": "str", "mistakes": {"008d4876a8eb4765b7d6e6eada374bb548f6202452c76cc617f403a56df54046": "il manque une des lignes fausses : il y en a exactement cinq", "0a33026ea63be4f1ab7683172488a637578438bf3b28c2330645307b61ba9ff4": "il manque une des lignes fausses : il y en a exactement cinq", "1cc70f76491935ff4b20516a422834245ff04274fcc5057af7772fda5ad285a8": "une ligne juste est citée à la place d'une ligne fausse", "1e9050faf72c3af009d7217ea6558a2c5ea8f4eda474a140274875f73b72d817": "il manque une des lignes fausses : il y en a exactement cinq", "37a2bb6b08fd4132675b63c99ce854163d5a266e2d1f96baebd0df54d90bbca1": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "390955e8903ef2e0e50a73873569e11caa84f8891ba84d5176d5998a1696b825": "une ligne juste est citée à la place d'une ligne fausse", "3a874824fea8e84d21aaf1624a571e12e33692854338b2bbd80d75e87cf

**CP5.10 — Problème de synthèse : un filtre récursif et un filtre à réponse finie, du signal à la stabilité**

In [11]:
wbm.record("CP5.10a", floats(H_10[:5]), decimals=6, mistakes={"dans hₖ = C Aᵏ B, B choisit une colonne de Aᵏ et C une ligne : vérifie lesquelles": floats(H12_10), "la réponse impulsionnelle commence à h₀ = C B": floats(H_10[1:6]), "hₖ est une sortie : relis quelle coordonnée de l'état C lit": floats(FIRST_10)})
wbm.record("CP5.10b", floats(OUT_10[1:6]), decimals=6, mistakes={"yₜ₊₁ = (h * u)ₜ : un pas de décalage entre la sortie et la convolution": CONV_10[1:6].tolist()})
wbm.record("CP5.10c", RHO_10, decimals=4, mistakes={"le module d'une valeur propre complexe tient compte de sa partie imaginaire": float(TR_10 / 2), "un module est une racine carrée : |λ|² = λ λ̄": float(DET_10)})
wbm.record("CP5.10d", NORMS_10, decimals=6, mistakes={"‖A‖₂ est une valeur singulière : la racine carrée d'une valeur propre de AᵀA": NORMS_10[:2] + [float(NORM2_10 ** 2)], "ta troisième valeur est-elle le plus grand facteur d'allongement ‖A x‖₂/‖x‖₂ ?": NORMS_10[:2] + [RHO_10], "ta troisième valeur est-elle bien le plus grand facteur d'allongement ‖A x‖₂/‖x‖₂ ?": NORMS_10[:2] + [round(RHO_10, 4)], "‖·‖₁ et ‖·‖∞ prennent la plus grande des sommes des valeurs absolues, colonne par colonne ou ligne par ligne": SIGNED_10 + NORMS_10[2:], "la norme de Frobenius n'est pas la norme induite ‖·‖₂": NORMS_10[:2] + [FROB_10], "la norme de Frobenius n'est pas la norme induite ‖·‖₂ : relis sa définition": NORMS_10[:2] + [round(FROB_10, 4)]})
wbm.record("CP5.10e", K_10, fractional="K est un nombre de termes : arrondis à l'entier qui garantit l'inégalité.", mistakes={"le plus petit K qui convient doit vérifier l'inégalité : arrondis vers le haut": K_10 - 1, "la majoration porte sur toute la queue, ‖A‖ᴷ/(1 − ‖A‖), avec une norme induite : le rayon spectral ne majore pas chaque pas": NO_FACTOR_10, "le rayon spectral ne majore pas chaque |hₖ| : la majoration se fait avec une norme induite, sur toute la queue": RHO_ONLY_10})
wbm.record("CP5.10f", float(WORST_10), decimals=6, mistakes={"avec K = 5, le filtre garde cinq termes de la réponse impulsionnelle, h₀ à h₄": float(WORST_SLIPS_10["four"]), "avec K = 5, le filtre garde les cinq premiers termes seulement": float(WORST_SLIPS_10["six"])})
wbm.record("CP5.10g", str(LIMIT_10), kind="fraction", simplified=True, mistakes={"l'équilibre vérifie x⋆ = A x⋆ + B u : la matrice du système est I − A, pas sa transposée": str(LIMIT_T_10), "la limite fait intervenir toute la réponse impulsionnelle, pas seulement ses premiers termes": str(PARTIAL_10)})

WBM_ANSWER {"array_rounded": {"1": "5431cb869d99175b162a0390a11f0030d7dc19b6d821820e39e542e9bcdec2aa", "2": "9ff65fd218c29e9971032024afd8fba5686463f67fdd7a8db26dc930b87cfca2", "3": "f5e673a4e8147c220a3c5ef74924d0fdddc9e742c0766c970d4b7c9980355097"}, "decimals": 6, "element_coarse": [["232f7d7cee15a36676bb73a9e9ba2b95fb885a47456ad527af146715fc2a0674"], ["cb2dd05b7b6e81924d661db580d0558d969d48f5fc5830274e99efc1a74415ae"], ["03a3b4c8816b02961d1428987f4f0e78539a235dc7d9a53dcbd84e38343bcc02"], ["e0b21782486dabbd8e7c2f43adb6706a23eb979cd8981d762d995e0abbbc853a"], ["cc90fc0d1d07e9a9a44ce05b6bc3051ef3be28b0c331b904b1ee359a73742359"]], "element_coarse_any": [[], ["d0ed06ce7450bbc7c509e63a15d9f6b84520306a0b021282ed09460fafd30325", "2352c129ffa630f7e35cf08b947d43850eef3581705c22e01cfc8af83aaaf97a", "054621ecc270e67b91e194fae3583041e5ae5fe048a8c5a043c584d01215de1b", "799ecdc81377ff59e4462b9bc144a9d2ebc2c3a6a9dcbb493b620940bcb49327"], ["1a65b314d162a4612b13cab577c44118ca8fbe821bdf03b6cb3668b670c5f5